<h1>Before we start</h1>
The apriori algorithm comes from the **mlxtend** library, which is not part of Anaconda. 
Run the cell below once to install it (about 30 seconds), then it is instant on every later run.

In [ ]:
# install the mlxtend library (first run only — skipped once it is installed)
try:
    import mlxtend
except ModuleNotFoundError:
    %pip install -q "mlxtend==0.23.4"

<h1>Introduction</h1>
https://github.com/luis-alarcon/Kaggle_BreadBasket

The dataset belongs to "The Bread Basket", a bakery in Edinburgh. It has 20,507 rows covering 9,465 transactions, with 5 columns:

- **Transaction**: transaction number. Rows sharing the same value belong to the same purchase — that is why there are fewer transactions than rows.
- **Item**: the product bought.
- **date_time**: date and time of the transaction (DD-MM-YYYY HH:MM).
- **period_day**: morning / afternoon / evening / night.
- **weekday_weekend**: whether the transaction fell on a weekday or the weekend.
<h2>Licence:</h2>
MIT


<h1>Step 1 Import the libraries </h1>

In [ ]:
# import the libraries we need:
# pandas      -> loading and reshaping the data
# matplotlib  -> plotting
# seaborn     -> nicer statistical plots (the bar chart in Step 5)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

<h1> Step 2 Import the data set </h1>

In [ ]:
# read the dataset
data = pd.read_csv("data/bread_basket.csv")

# show the first 5 rows to check it loaded correctly
data.head()

In [ ]:
# summary statistics — only Transaction is numeric, so only it appears
# count = number of rows (20,507); max = highest transaction number (9,684)
data.describe()

<h1> Step 3 Converting Date Time into the correct format </h1>

In [ ]:
# convert the date_time text into a real datetime value
# the text is day-month-year (30-10-2016 09:58), so we tell pandas the exact format
data['date_time'] = pd.to_datetime(data['date_time'], format='%d-%m-%Y %H:%M')

In [ ]:
# look at the first 10 rows — date_time is now a proper datetime
data.head(10)


<h1>Step 4 Pre-processing the data </h1>
In this step we break the datetime into useful parts (month, hour, weekday) and tidy the item names, so the data is in the format that apriori will accept.

In [ ]:
# break the datetime into separate parts we can analyse

data['date'] = data['date_time'].dt.date   # just the date
data['time'] = data['date_time'].dt.time   # just the time

# .dt.month gives the month as a number (1-12);
# .map() converts each number to its month name
data['month'] = data['date_time'].dt.month.map({
    1:'January', 2:'February', 3:'March', 4:'April', 5:'May', 6:'June',
    7:'July', 8:'August', 9:'September', 10:'October', 11:'November', 12:'December'})

# .dt.hour gives the hour (0-23); map each hour to a readable time band
data['hour'] = data['date_time'].dt.hour.map({
    1:'1-2', 7:'7-8', 8:'8-9', 9:'9-10', 10:'10-11', 11:'11-12', 12:'12-13',
    13:'13-14', 14:'14-15', 15:'15-16', 16:'16-17', 17:'17-18', 18:'18-19',
    19:'19-20', 20:'20-21', 21:'21-22', 22:'22-23', 23:'23-24'})

# .dt.weekday gives the day as a number (0 = Monday); map it to the day name
data['weekday'] = data['date_time'].dt.weekday.map({
    0:'Monday', 1:'Tuesday', 2:'Wednesday', 3:'Thursday', 4:'Friday', 5:'Saturday', 6:'Sunday'})

# the original date_time column has done its job — drop it
data.drop('date_time', axis=1, inplace=True)



In [ ]:
# tidy the item names so identical products always match:
# .str.strip() removes stray spaces around the name
# .str.lower() makes everything lowercase ("Coffee" and "coffee" become one item)
data['Item'] = data['Item'].str.strip()
data['Item'] = data['Item'].str.lower()

<h1>Step 5 Visualising the data </h1>


In [ ]:
# count how many times each item was sold, keep the top 20
top20 = data.Item.value_counts().head(20)

# bar chart of the top 20 items
plt.figure(figsize=(20,5))
sns.barplot(x=top20.index, y=top20.values, hue=top20.index, palette='pastel', legend=False)
plt.xlabel('Items', size=15)
plt.xticks(rotation=45)          # angle the labels so they don't overlap
plt.ylabel('Count', size=15)
plt.title('20 Most Popular Items', color='navy', size=20)
plt.show()

In [ ]:
# count how many of each item appears in each transaction
# groupby collects the rows for every (Transaction, Item) pair;
# .count() counts them; reset_index turns the result back into a table
data = data.groupby(['Transaction','Item'])['Item'].count().reset_index(name='Count')
data

In [ ]:
# reshape into "basket" format: one ROW per transaction, one COLUMN per item,
# and the count of that item in that transaction as the value
# fillna(0) puts a 0 where an item was not part of the transaction
basket = data.pivot_table(index='Transaction', columns='Item', values='Count', aggfunc='sum').fillna(0)
basket.head()

<h1>Step 6 Encoding the data </h1>
In this step we convert the transactions into binary: **True** if the item was bought in that transaction, **False** if it was not. The apriori implementation expects exactly this True/False format.

In [ ]:
# True where the item was bought at least once in that transaction, False otherwise
basket_sets = basket > 0
basket_sets.head()

<h1>Step 7 Apply the Apriori </h1>
In this step we use the apriori algorithm to find the frequent itemsets — the combinations of items that are often bought together. It comes from the mlxtend library we installed in Step 0. To explore the library further: https://rasbt.github.io/mlxtend/


In [ ]:
# import the two functions we need from mlxtend
from mlxtend.frequent_patterns import association_rules, apriori

# find every set of items that appears together in at least 1% of transactions
# min_support=0.01  -> the 1% cut-off
# use_colnames=True -> show item names instead of column numbers
frequent_sets = apriori(basket_sets, min_support=0.01, use_colnames=True)
frequent_sets


<h1>Step 8 Apply Association rules </h1>
Now we turn the frequent itemsets into "if X then Y" rules, keep only rules with lift above 1, and sort them from highest to lowest confidence.

## How to read the rules
- **antecedent → consequent**: "customers who bought X also bought Y"
- **support**: how often the whole combination appears across all transactions
- **confidence**: of the transactions containing X, the fraction that also contain Y
- **lift**: how much more likely Y is when X is present, compared to Y's normal rate.

Lift above 1 means the items genuinely go together, not by coincidence.

In [ ]:
# turn the frequent itemsets into "if X then Y" rules
# metric="lift", min_threshold=1 -> only keep rules where the pairing is
# more likely than chance (lift above 1)
rules = association_rules(frequent_sets, metric="lift", min_threshold=1)

# sort so the most reliable rules (highest confidence) appear first
rules.sort_values('confidence', ascending=False, inplace=True)
rules

<h1>What did we learn?</h1>

- **The data needed reshaping, not just cleaning.** Apriori doesn't want rows of items — it wants one row per transaction with True/False per product. Most of this lab was building that basket format.
- **Coffee anchors the shop.** It appears in 48% of all transactions — so almost every strong rule points *at* coffee.
- **The best rule is toast → coffee.** 70% of transactions with toast also include coffee (confidence 0.70), which is 1.47× coffee's normal rate (lift). Someone settling in for breakfast buys both.
- **Confidence needs lift beside it.** "X → coffee" rules have high confidence partly because coffee is everywhere. Lift above 1 is what tells us the pairing is real rather than coffee just being popular.
- **What would a shop do with this?** Put the toast station near the coffee machine, bundle a breakfast deal, and never let the espresso machine break on a Saturday morning.